# Обрезка праймеров — человек (`PRJEB40348`)

Вход — стадия `trimmed` после `cutadapt` (только технические key/MID + Illumina-адаптеры) и `fastp -q 30 -u 40 -l 250`. `MaskPrimers.py align --mode cut` снимает праймеры. Последовательности праймеров заданы прямо в ноутбуке (dict `PRIMERS`), внешние FASTA-файлы не нужны. Стадия `pr_trimmed` заменяется атомарно только после обработки всех файлов.


In [ ]:
import os, sys, sysconfig, shutil, subprocess, time
from pathlib import Path
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [_CONDA_ENV + "/lib/python3.11/site-packages", sysconfig.get_path("purelib")]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)
print("MaskPrimers.py:", shutil.which("MaskPrimers.py"))


In [ ]:
# Праймеры человека — универсальный набор FR1/J из Cheng 2011.
# Источник: Table S1 связанного расширенного проекта E-MTAB-10859 / PRJEB47001
# (Lomakin et al. 2022, PMC9425031); 68 из 70 FASTQ PRJEB40348 совпадают с ним по MD5.
# Ориентация: прямые — VH/Vk/Vl FR1, обратные — JH для ампликонов с J-конца.
PRIMERS = {
    'VH1_1': 'CAGGTCCAGCTTGTGCAGTCTGG',
    'VH1_2': 'CAGGTCCAGCTKGTGCAGTCTGG',
    'VH1_3': 'CAGATCCAGCTGGTGCAGTCTGG',
    'VH2_1': 'CAGATCACCTTGAAGGAGTCTGG',
    'VH3_1': 'GAGGTGCAGCTGGTGGAGTCTGG',
    'VH3_2': 'GAGGTGCAGCTGGTGGAGTCTGGG',
    'VH4_1': 'CAGGTGCAGCTACAGCAGTGG',
    'VH4_2': 'CAGGTGCAGCTACAGCAATGGG',
    'VH5_1': 'GAGGTGCAGCTGTTGCAGTCTGC',
    'VH6_1': 'CAGGTACAGCTGCAGCAGTCAG',
    'VH7_1': 'CAGGTGCAASTGGTGCAATCTGG',
    'Vk1_1': 'GACATCCAGATGACCCAGTCTCC',
    'Vk1_2': 'GACATCCAGTTGACCCAGTCTCC',
    'Vk2_1': 'GATGTTGTGATGACTCAGTCTCC',
    'Vk2_2': 'GATATTGTGATGACTCAGTCTCC',
    'Vk3_1': 'GAAATTGTGTTGACGCAGTCTCC',
    'Vk4_1': 'GACATCGTGATGACCCAGTCTCC',
    'Vk5_1': 'GAAACGACACTCACGCAGTCTCC',
    'Vk6_1': 'GAAATTGTGCTGACTCAGTCTCC',
    'Vk7_1': 'GACATTGTGATGACCCAGTCTCC',
    'Vl1_1': 'CAGTCTGTGCTGACTCAGCCACC',
    'Vl1_2': 'CAGTCTGTGCTGACACAGCCACC',
    'Vl2_1': 'CAGTCTGCCCTGACTCAGCCT',
    'Vl3_1': 'TCCTATGTGCTGACTCAGCCACC',
    'Vl3_2': 'TCTTCTGAGCTGACTCAGGACCC',
    'Vl4_1': 'CAGTCTGTGCTGACTCAGCCGC',
    'Vl5_1': 'CAGCCTGTGCTGACTCAGCCT',
    'Vl6_1': 'AATTTTATGCTGACTCAGCCCC',
    'Vl7_1': 'CAGRCTGTGGTGACTCAGGAGCC',
    'Vl8_1': 'CAGACTGTGGTGACCCAGGAGCC',
    'Vl9_1': 'CAGCCTGTGCTGACTCAGCCTTC',
    'Vl10_1': 'CAGCCAGGGCTGACTCAGCCT',
    'JH1_rev': 'TGAGGAGACGGTGACCAGGGT',
    'JH2_rev': 'TGAGGAGACGGTGACCATGGT',
    'JH3_rev': 'TGAGGAGACGGTGACCAGGGT',
    'JH4_rev': 'TGAGGAGACGGTGACCATGGT',
}
print('human: 36 праймеров задано в PRIMERS')


In [ ]:
DATASET = 'PRJEB40348'

def _tool(name):
    path = shutil.which(name)
    if not path:
        raise FileNotFoundError(name)
    return path

def _run_visible(cmd, stdout_log, stderr_log, heartbeat=30):
    started = time.monotonic()
    print('[run]', ' '.join(map(str, cmd)), flush=True)
    with open(stdout_log, 'w') as stdout, open(stderr_log, 'w') as stderr:
        proc = subprocess.Popen([str(x) for x in cmd], stdout=stdout, stderr=stderr, text=True)
        print(f'PID={proc.pid}', flush=True)
        while proc.poll() is None:
            print(f'PID={proc.pid} elapsed={(time.monotonic()-started)/60:.1f}min', flush=True)
            time.sleep(heartbeat)
    if proc.returncode:
        raise RuntimeError(f'rc={proc.returncode}; see {stderr_log}')

def _write_primer_fasta(primers, path):
    """Праймеры заданы прямо в ноутбуке; FASTA создаётся на время запуска."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as handle:
        for name, sequence in primers.items():
            handle.write(f'>{name}\n{sequence}\n')
    return path

def _promote(staging, final):
    previous = final.parent / f'.{final.name}.previous'
    if previous.exists():
        shutil.rmtree(previous)
    if final.exists():
        final.rename(previous)
    try:
        staging.rename(final)
    except Exception:
        if previous.exists() and not final.exists():
            previous.rename(final)
        raise
    if previous.exists():
        shutil.rmtree(previous)

def run_primer_trim(volume, dataset=DATASET, force=False):
    vol = Path(volume)
    src = vol / 'results' / dataset / 'trimmed' / 'fastq'
    final = vol / 'results' / dataset / 'pr_trimmed'
    staging = final.parent / '.pr_trimmed.staging'
    inputs = sorted(src.glob('*.trim.fastq.gz'))
    if not inputs:
        raise FileNotFoundError(f'No trimmed FASTQ in {src}')
    if staging.exists():
        if not force:
            raise FileExistsError(staging)
        shutil.rmtree(staging)
    out = staging / 'fastq'
    logs = staging / 'maskprimer_logs'
    work = staging / 'work'
    refs = staging / 'primer_refs'
    for d in (out, logs, work, refs):
        d.mkdir(parents=True, exist_ok=True)
    primer_fasta = _write_primer_fasta(PRIMERS, refs / f'{dataset}_primers.fasta')
    print(f'[primer_trim] {dataset}: {len(inputs)} files; {len(PRIMERS)} праймеров встроено в ноутбук', flush=True)
    for f in inputs:
        stem = f.name.removesuffix('.trim.fastq.gz')
        outname = stem + '.pr'
        cmd = [_tool('MaskPrimers.py'), 'align', '-s', f, '-p', primer_fasta, '--mode', 'cut',
               '--maxerror', '0.2', '--maxlen', '50', '--nproc', '4',
               '--outdir', work, '--outname', outname]
        _run_visible(cmd, logs / f'{stem}.stdout.log', logs / f'{stem}.stderr.log')
        candidates = [work / f'{outname}_primers-pass.fastq.gz', work / f'{outname}_primers-pass.fastq']
        produced = next((p for p in candidates if p.is_file()), None)
        if produced is None:
            raise RuntimeError(f'No primer-pass output for {f.name}')
        target = out / f'{stem}.pr.fastq.gz'
        if produced.suffix == '.gz':
            produced.rename(target)
        else:
            import gzip
            with open(produced, 'rb') as source, gzip.open(target, 'wb', compresslevel=1) as dest:
                shutil.copyfileobj(source, dest)
            produced.unlink()
    if len(list(out.glob('*.pr.fastq.gz'))) != len(inputs):
        raise RuntimeError('Primer output completeness validation failed; pr_trimmed не заменён')
    shutil.rmtree(work)
    _promote(staging, final)
    print(f'[primer_trim] DONE and promoted: {final}', flush=True)


## Запуск с атомарной заменой стадии


In [ ]:
run_primer_trim('/data/user/epishkin', 'PRJEB40348', force=True)
